In [1]:
# Autoreload magic: reload modules automatically before executing code
%load_ext autoreload
%autoreload 2

# Full-Text Schema Extraction Without Vector DB

Test whether feeding complete document sections directly to GPT (without embeddings or chunking) improves Fuster metadata extraction quality over abstract-only baseline.

**Plan**: `plans/plan-fullTextSchemaExtractionWithoutVectorDb.md`

## Approach
- Parse PDF with GROBID to extract hierarchical sections
- Select relevant sections (Methods, Study Area, Data, Species, etc.)
- Concatenate into a single prompt (no vector DB, no chunking)
- Compare extraction quality vs abstract-only baseline

In [2]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import pandas as pd
import re
from dotenv import load_dotenv

load_dotenv()

# Project modules
from llm_metadata.pdf_parsing import process_pdf, ParsedDocument, Section
from llm_metadata.section_normalize import classify_section
from llm_metadata.chunking import count_tokens
from llm_metadata.gpt_classify import classify_abstract
from llm_metadata.schemas.fuster_features import DatasetFeatureExtraction
from llm_metadata.schemas.evaluation import (
    evaluate_indexed, EvaluationConfig, micro_average, macro_f1
)
from llm_metadata.schemas.chunk_metadata import SectionType
from llm_metadata.schemas.validation import DataFrameValidator

# Test DOI mapping (from manifest.csv)
TEST_DOI = "10.5061/dryad.3nh72"  # Dataset DOI
ARTICLE_DOI = "10.1111/ddi.12496"  # Corresponding article DOI

PDF_PATH = Path("data/pdfs/fuster") / f"{ARTICLE_DOI.replace('/', '_')}.pdf"
print(f"PDF path: {PDF_PATH.absolute()}")
print(f"PDF exists: {PDF_PATH.exists()}")
print(f"PDF size: {PDF_PATH.stat().st_size / 1024:.1f} KB" if PDF_PATH.exists() else "")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
PDF path: c:\Users\beav3503\dev\llm_metadata\data\pdfs\fuster\10.1111_ddi.12496.pdf
PDF exists: True
PDF size: 1257.7 KB


## Step 1: Parse PDF with GROBID

Use GROBID to extract document structure with hierarchical sections.

In [2]:
# Parse PDF with GROBID
work_id = ARTICLE_DOI.replace('/', '_')

print("Parsing PDF with GROBID...")
tei_path, doc = process_pdf(
    pdf_path=PDF_PATH,
    work_id=work_id,
    grobid_url="http://localhost:8070"
)

print(f"\n" + "="*60)
print("DOCUMENT PARSED")
print("="*60)
print(f"TEI XML: {tei_path}")
print(f"Title: {doc.title}")
print(f"Language: {doc.language}")
print(f"Keywords: {doc.keywords}")
print(f"Abstract: {len(doc.abstract)} chars" if doc.abstract else "No abstract")
print(f"Sections: {len(doc.sections)} top-level")

Parsing PDF with GROBID...

DOCUMENT PARSED
TEI XML: C:\Users\beav3503\dev\llm_metadata\artifacts\tei\10.1111_ddi.12496.grobid.tei.xml
Title: Patchy distribution and low effective population size raise concern for an at-risk top predator
Language: en
Keywords: ['conservation genetics', 'Eastern Wolf', 'effective population size', 'predator distribution']
Abstract: 2078 chars
Sections: 7 top-level


In [3]:
# Explore section structure (from pdf_chunking_exploration.ipynb pattern)
def print_section_tree(sections, level=0):
    for sec in sections:
        indent = "  " * level
        section_type = classify_section(sec.title)
        text_len = len(sec.text) if sec.text else 0
        tokens = count_tokens(sec.text) if sec.text else 0
        print(f"{indent}[{section_type.value:12}] {sec.title[:50]:50} ({tokens:4} tokens)")
        if sec.subsections:
            print_section_tree(sec.subsections, level + 1)

print("\nDocument Section Tree:")
print("="*80)
print_section_tree(doc.sections)


Document Section Tree:
[INTRO       ] INTRODUCTION                                       (1375 tokens)
[METHODS     ] METHODS                                            (   0 tokens)
[OTHER       ] Sample collection, preparation and amplification   ( 860 tokens)
[OTHER       ] Estimating genotyping error                        ( 238 tokens)
[OTHER       ] General statistics, assignment tests and effective (1008 tokens)
[RESULTS     ] RESULTS                                            ( 704 tokens)
[DISCUSSION  ] DISCUSSION                                         (1630 tokens)


## Step 2: Select Relevant Sections

Filter sections by:
- Section type: ABSTRACT, METHODS
- Keywords in title: data, dataset, survey, site, area, species, sampling, collection, study

In [ ]:
# Section selection criteria from plan
RELEVANT_TYPES = {SectionType.ABSTRACT, SectionType.METHODS}
KEYWORD_PATTERN = re.compile(
    r'\b(data|dataset|survey|site|area|species|sampling|collection|study|material|method|sample)\b',
    re.IGNORECASE
)

def is_relevant_section(section: Section) -> bool:
    """Check if section is relevant for metadata extraction."""
    section_type = classify_section(section.title)
    
    # Check by section type
    if section_type in RELEVANT_TYPES:
        return True
    
    # Check by keyword in title
    if KEYWORD_PATTERN.search(section.title):
        return True
    
    return False

def collect_relevant_sections(sections, parent_relevant=False):
    """Recursively collect relevant sections."""
    relevant = []
    for sec in sections:
        is_rel = is_relevant_section(sec) or parent_relevant
        if is_rel and sec.text and sec.text.strip():
            relevant.append(sec)
        # Recurse into subsections (inherit parent relevance)
        relevant.extend(collect_relevant_sections(sec.subsections, is_rel))
    return relevant

relevant_sections = collect_relevant_sections(doc.sections)

print(f"Found {len(relevant_sections)} relevant sections:")
print("="*70)
for sec in relevant_sections:
    section_type = classify_section(sec.title)
    tokens = count_tokens(sec.text)
    print(f"  [{section_type.value:12}] {sec.title[:40]:40} ({tokens:4} tokens)")

Found 1 relevant sections:
  [OTHER       ] Sample collection, preparation and ampli ( 860 tokens)


## Step 3: Token Analysis

In [5]:
# Token counts
token_data = []

# Abstract
if doc.abstract:
    token_data.append({
        'section': 'Abstract',
        'type': 'ABSTRACT',
        'chars': len(doc.abstract),
        'tokens': count_tokens(doc.abstract)
    })

# Relevant sections
for sec in relevant_sections:
    section_type = classify_section(sec.title)
    token_data.append({
        'section': sec.title[:40],
        'type': section_type.value,
        'chars': len(sec.text),
        'tokens': count_tokens(sec.text)
    })

token_df = pd.DataFrame(token_data)
token_df['cumulative_tokens'] = token_df['tokens'].cumsum()

print("Token Usage by Section:")
display(token_df)

total_tokens = token_df['tokens'].sum()
abstract_tokens = count_tokens(doc.abstract) if doc.abstract else 0

print(f"\n" + "="*50)
print(f"Abstract-only tokens:  {abstract_tokens:,}")
print(f"Full-text tokens:      {total_tokens:,}")
print(f"Token increase:        {total_tokens - abstract_tokens:,} ({(total_tokens/abstract_tokens - 1)*100:.0f}% more)" if abstract_tokens else "N/A")
print(f"Context limit check:   {'OK' if total_tokens < 80000 else 'EXCEEDS 80K LIMIT'}")

Token Usage by Section:


,section,type,chars,tokens,cumulative_tokens
0,Abstract,ABSTRACT,2078,370,370
1,"Sample collection, preparation and ampli",OTHER,3631,860,1230



Abstract-only tokens:  370
Full-text tokens:      1,230
Token increase:        860 (232% more)
Context limit check:   OK


## Step 4: Build Full-Text Prompt

In [6]:
def build_fulltext_prompt(doc: ParsedDocument, sections: list) -> str:
    """Build full-text prompt with abstract and relevant sections."""
    parts = []
    
    # Abstract first
    if doc.abstract:
        parts.append("## Abstract\n")
        parts.append(doc.abstract)
        parts.append("\n\n")
    
    # Relevant sections with headers
    for sec in sections:
        parts.append(f"## {sec.title}\n")
        parts.append(sec.text)
        parts.append("\n\n")
    
    return "".join(parts)

fulltext_prompt = build_fulltext_prompt(doc, relevant_sections)
fulltext_tokens = count_tokens(fulltext_prompt)

print(f"Full-text prompt: {len(fulltext_prompt):,} chars, {fulltext_tokens:,} tokens")
print("\n" + "="*70)
print("PROMPT PREVIEW (first 1500 chars):")
print("="*70)
print(fulltext_prompt[:1500])
print("\n[...]")

Full-text prompt: 5,777 chars, 1,242 tokens

PROMPT PREVIEW (first 1500 chars):
## Abstract
Aim Understanding carnivore distribution is important for management decisions that aim to restore naturally regulated ecosystems and preserve biodiversity. Eastern Wolves, a species at risk in Canada, are centralized in Algonquin Provincial Park and their ability to disperse and establish themselves elsewhere is limited by human-caused mortality associated with hunting, trapping and vehicle collisions. Here, we refine our understanding of Eastern Wolf distribution and provide the first estimates of their effective population size.

Location Southern Ontario and Gatineau Quebec.

Methods We used non-invasive samples, as well as blood samples archived from other research projects, collected between 2010 and 2014 to generate autosomal microsatellite genotypes at 12 loci for 98 Canis individuals. We utilized Bayesian and multivariate clustering analyses to identify Eastern Wolves in regions that we

## Step 5: Load Ground Truth

In [7]:
# Load manual annotations
ANNOT_FILE = Path("data/dataset_092624.xlsx")
df = pd.read_excel(ANNOT_FILE)

# Find row for test DOI
test_row = df[df['url'].str.contains(TEST_DOI, na=False)]
print(f"Found {len(test_row)} rows for DOI {TEST_DOI}")

# Validate to Pydantic model
relevant_cols = ['url', 'title', 'full_text', 'data_type', 'geospatial_info_dataset', 
                 'spatial_range_km2', 'temporal_range', 'temp_range_i', 'temp_range_f', 
                 'species', 'referred_dataset']

test_df = test_row[relevant_cols]
validator = DataFrameValidator(DatasetFeatureExtraction)
report = validator.validate(test_df)

ground_truth = report.valid_rows[0]
print("\n" + "="*50)
print("GROUND TRUTH")
print("="*50)
print(ground_truth.model_dump_json(indent=2))

2026-01-09 09:31:26.088 | INFO     | llm_metadata.schemas.validation:validate:233 - Validation complete: 1 valid, 0 invalid, 0 errors


Found 1 rows for DOI 10.5061/dryad.3nh72

GROUND TRUTH
{
  "data_type": [
    "presence-only",
    "genetic_analysis"
  ],
  "geospatial_info_dataset": [
    "sample"
  ],
  "spatial_range_km2": null,
  "temporal_range": "2010-2014",
  "temp_range_i": 2010,
  "temp_range_f": 2014,
  "species": [
    "Eastern coyote",
    "eastern wolf"
  ],
  "referred_dataset": null,
  "valid_yn": null,
  "reason_not_valid": null
}


## Step 6: Run Extractions (Full-text vs Abstract-only)

In [8]:
# Full-text extraction
print("Running FULL-TEXT extraction...")
print(f"  Input: {fulltext_tokens:,} tokens")

fulltext_result = classify_abstract(
    abstract=fulltext_prompt,
    text_format=DatasetFeatureExtraction,
    model="gpt-5-mini",
    reasoning={"effort": "low"},
    max_output_tokens=4096,
)

fulltext_extraction = fulltext_result['output']
fulltext_usage = fulltext_result['response'].usage

print(f"\nToken Usage:")
print(f"  Input:  {fulltext_usage.input_tokens:,}")
print(f"  Output: {fulltext_usage.output_tokens:,}")

# Cost (GPT-5-mini: $0.25/1M input, $2.005/1M output)
ft_input_cost = (fulltext_usage.input_tokens / 1_000_000) * 0.25
ft_output_cost = (fulltext_usage.output_tokens / 1_000_000) * 2.005
ft_total_cost = ft_input_cost + ft_output_cost
print(f"  Cost:   ${ft_total_cost:.5f}")

print("\n" + "="*50)
print("FULL-TEXT EXTRACTION:")
print("="*50)
print(fulltext_extraction.model_dump_json(indent=2))

Running FULL-TEXT extraction...
  Input: 1,242 tokens

Token Usage:
  Input:  2,158
  Output: 1,339
  Cost:   $0.00322

FULL-TEXT EXTRACTION:
{
  "data_type": [
    "genetic_analysis",
    "distribution"
  ],
  "geospatial_info_dataset": [
    "site",
    "administrative_units",
    "distribution"
  ],
  "spatial_range_km2": null,
  "temporal_range": "2010 to 2014; additional samples from 2003 to 2005 and 2013 to 2014",
  "temp_range_i": 2003,
  "temp_range_f": 2014,
  "species": [
    "98 Canis individuals",
    "34 individuals as Eastern Wolves",
    "Eastern Coyotes",
    "admixed among the different Canis types",
    "Eastern Wolves"
  ],
  "referred_dataset": null,
  "valid_yn": null,
  "reason_not_valid": null
}


In [9]:
# Abstract-only extraction
abstract_prompt = doc.abstract if doc.abstract else ""
abstract_tokens = count_tokens(abstract_prompt)

print("Running ABSTRACT-ONLY extraction...")
print(f"  Input: {abstract_tokens:,} tokens")

abstract_result = classify_abstract(
    abstract=abstract_prompt,
    text_format=DatasetFeatureExtraction,
    model="gpt-5-mini",
    reasoning={"effort": "low"},
    max_output_tokens=4096,
)

abstract_extraction = abstract_result['output']
abstract_usage = abstract_result['response'].usage

print(f"\nToken Usage:")
print(f"  Input:  {abstract_usage.input_tokens:,}")
print(f"  Output: {abstract_usage.output_tokens:,}")

ab_input_cost = (abstract_usage.input_tokens / 1_000_000) * 0.25
ab_output_cost = (abstract_usage.output_tokens / 1_000_000) * 2.005
ab_total_cost = ab_input_cost + ab_output_cost
print(f"  Cost:   ${ab_total_cost:.5f}")

print("\n" + "="*50)
print("ABSTRACT-ONLY EXTRACTION:")
print("="*50)
print(abstract_extraction.model_dump_json(indent=2))

Running ABSTRACT-ONLY extraction...
  Input: 370 tokens

Token Usage:
  Input:  1,301
  Output: 1,378
  Cost:   $0.00309

ABSTRACT-ONLY EXTRACTION:
{
  "data_type": [
    "genetic_analysis",
    "distribution",
    "presence-absence"
  ],
  "geospatial_info_dataset": [
    "sample",
    "site",
    "range",
    "distribution",
    "administrative_units"
  ],
  "spatial_range_km2": null,
  "temporal_range": "2010 to 2014",
  "temp_range_i": 2010,
  "temp_range_f": 2014,
  "species": [
    "Eastern Wolves",
    "Eastern Coyotes",
    "Canis individuals"
  ],
  "referred_dataset": null,
  "valid_yn": null,
  "reason_not_valid": null
}


## Step 7: Evaluate Both Approaches

In [10]:
# Evaluation fields
eval_fields = ['data_type', 'geospatial_info_dataset', 'spatial_range_km2', 
               'temporal_range', 'temp_range_i', 'temp_range_f', 'species']

# Evaluate full-text
fulltext_report = evaluate_indexed(
    true_by_id={TEST_DOI: ground_truth},
    pred_by_id={TEST_DOI: fulltext_extraction},
    fields=eval_fields,
    config=EvaluationConfig(treat_lists_as_sets=True)
)

# Evaluate abstract-only
abstract_report = evaluate_indexed(
    true_by_id={TEST_DOI: ground_truth},
    pred_by_id={TEST_DOI: abstract_extraction},
    fields=eval_fields,
    config=EvaluationConfig(treat_lists_as_sets=True)
)

# Aggregate metrics
ft_micro = micro_average(fulltext_report.field_metrics.values())
ft_macro = macro_f1(fulltext_report.field_metrics.values())
ab_micro = micro_average(abstract_report.field_metrics.values())
ab_macro = macro_f1(abstract_report.field_metrics.values())

print("FULL-TEXT Per-field Metrics:")
display(fulltext_report.metrics_to_pandas())

print("\nABSTRACT-ONLY Per-field Metrics:")
display(abstract_report.metrics_to_pandas())

FULL-TEXT Per-field Metrics:


,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
0,data_type,1,1,1,0,1,0.5,0.5,0.5,1.0,0.0
1,geospatial_info_dataset,0,3,1,0,1,0.0,0.0,NaN,0.0,0.0
2,spatial_range_km2,0,0,0,1,1,NaN,NaN,NaN,1.0,1.0
6,species,0,5,2,0,1,0.0,0.0,NaN,0.0,0.0
5,temp_range_f,1,0,0,0,1,1.0,1.0,1.0,1.0,1.0
4,temp_range_i,0,1,1,0,1,0.0,0.0,NaN,0.0,0.0
3,temporal_range,0,1,1,0,1,0.0,0.0,NaN,0.0,0.0



ABSTRACT-ONLY Per-field Metrics:


,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
0,data_type,1,2,1,0,1,0.333333,0.5,0.400000,1.0,0.0
1,geospatial_info_dataset,1,4,0,0,1,0.200000,1.0,0.333333,1.0,0.0
2,spatial_range_km2,0,0,0,1,1,NaN,NaN,NaN,1.0,1.0
6,species,0,3,2,0,1,0.000000,0.0,NaN,0.0,0.0
5,temp_range_f,1,0,0,0,1,1.000000,1.0,1.000000,1.0,1.0
4,temp_range_i,1,0,0,0,1,1.000000,1.0,1.000000,1.0,1.0
3,temporal_range,0,1,1,0,1,0.000000,0.0,NaN,0.0,0.0


In [11]:
# Side-by-side comparison
comparison_data = []
for field in eval_fields:
    ft_m = fulltext_report.field_metrics.get(field)
    ab_m = abstract_report.field_metrics.get(field)
    
    ft_f1 = ft_m.f1 if ft_m and ft_m.f1 else 0
    ab_f1 = ab_m.f1 if ab_m and ab_m.f1 else 0
    
    comparison_data.append({
        'field': field,
        'fulltext_f1': round(ft_f1, 3),
        'abstract_f1': round(ab_f1, 3),
        'delta': round(ft_f1 - ab_f1, 3),
        'winner': 'FULL-TEXT' if ft_f1 > ab_f1 else ('ABSTRACT' if ab_f1 > ft_f1 else 'TIE')
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*70)
print("COMPARISON: FULL-TEXT vs ABSTRACT-ONLY (per field)")
print("="*70)
display(comparison_df)

# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"{'Metric':<25} {'Full-text':>12} {'Abstract':>12} {'Delta':>12}")
print("-"*70)
print(f"{'Micro Precision':<25} {ft_micro.precision or 0:>12.3f} {ab_micro.precision or 0:>12.3f} {(ft_micro.precision or 0) - (ab_micro.precision or 0):>+12.3f}")
print(f"{'Micro Recall':<25} {ft_micro.recall or 0:>12.3f} {ab_micro.recall or 0:>12.3f} {(ft_micro.recall or 0) - (ab_micro.recall or 0):>+12.3f}")
print(f"{'Micro F1':<25} {ft_micro.f1 or 0:>12.3f} {ab_micro.f1 or 0:>12.3f} {(ft_micro.f1 or 0) - (ab_micro.f1 or 0):>+12.3f}")
print(f"{'Macro F1':<25} {ft_macro or 0:>12.3f} {ab_macro or 0:>12.3f} {(ft_macro or 0) - (ab_macro or 0):>+12.3f}")
print("-"*70)
print(f"{'Input Tokens':<25} {fulltext_usage.input_tokens:>12,} {abstract_usage.input_tokens:>12,} {fulltext_usage.input_tokens - abstract_usage.input_tokens:>+12,}")
print(f"{'Cost ($)':<25} {ft_total_cost:>12.5f} {ab_total_cost:>12.5f} {ft_total_cost - ab_total_cost:>+12.5f}")
print("="*70)

# Verdict
ft_wins = sum(1 for r in comparison_data if r['winner'] == 'FULL-TEXT')
ab_wins = sum(1 for r in comparison_data if r['winner'] == 'ABSTRACT')
ties = sum(1 for r in comparison_data if r['winner'] == 'TIE')
print(f"\nFields won: FULL-TEXT={ft_wins}, ABSTRACT={ab_wins}, TIE={ties}")


COMPARISON: FULL-TEXT vs ABSTRACT-ONLY (per field)


,field,fulltext_f1,abstract_f1,delta,winner
0,data_type,0.5,0.400,0.100,FULL-TEXT
1,geospatial_info_dataset,0.0,0.333,-0.333,ABSTRACT
2,spatial_range_km2,0.0,0.000,0.000,TIE
3,temporal_range,0.0,0.000,0.000,TIE
4,temp_range_i,0.0,1.000,-1.000,ABSTRACT
5,temp_range_f,1.0,1.000,0.000,TIE
6,species,0.0,0.000,0.000,TIE



SUMMARY
Metric                       Full-text     Abstract        Delta
----------------------------------------------------------------------
Micro Precision                  0.154        0.286       -0.132
Micro Recall                     0.250        0.500       -0.250
Micro F1                         0.190        0.364       -0.173
Macro F1                         0.750        0.683       +0.067
----------------------------------------------------------------------
Input Tokens                     2,158        1,301         +857
Cost ($)                       0.00322      0.00309     +0.00014

Fields won: FULL-TEXT=1, ABSTRACT=2, TIE=4


## Step 8: Detailed Error Analysis

In [12]:
# Show side-by-side values for each field
print("="*80)
print("FIELD-BY-FIELD COMPARISON")
print("="*80)

gt_dict = ground_truth.model_dump()
ft_dict = fulltext_extraction.model_dump()
ab_dict = abstract_extraction.model_dump()

for field in eval_fields:
    print(f"\n{field}:")
    print(f"  Ground Truth:  {gt_dict.get(field)}")
    print(f"  Full-text:     {ft_dict.get(field)}")
    print(f"  Abstract-only: {ab_dict.get(field)}")
    
    # Check match
    ft_match = fulltext_report.field_metrics[field].exact_match_rate == 1.0 if field in fulltext_report.field_metrics else False
    ab_match = abstract_report.field_metrics[field].exact_match_rate == 1.0 if field in abstract_report.field_metrics else False
    print(f"  Match? FT={ft_match}, AB={ab_match}")

FIELD-BY-FIELD COMPARISON

data_type:
  Ground Truth:  ['presence-only', 'genetic_analysis']
  Full-text:     ['genetic_analysis', 'distribution']
  Abstract-only: ['genetic_analysis', 'distribution', 'presence-absence']
  Match? FT=False, AB=False

geospatial_info_dataset:
  Ground Truth:  ['sample']
  Full-text:     ['site', 'administrative_units', 'distribution']
  Abstract-only: ['sample', 'site', 'range', 'distribution', 'administrative_units']
  Match? FT=False, AB=False

spatial_range_km2:
  Ground Truth:  None
  Full-text:     None
  Abstract-only: None
  Match? FT=True, AB=True

temporal_range:
  Ground Truth:  2010-2014
  Full-text:     2010 to 2014; additional samples from 2003 to 2005 and 2013 to 2014
  Abstract-only: 2010 to 2014
  Match? FT=False, AB=False

temp_range_i:
  Ground Truth:  2010
  Full-text:     2003
  Abstract-only: 2010
  Match? FT=False, AB=True

temp_range_f:
  Ground Truth:  2014
  Full-text:     2014
  Abstract-only: 2014
  Match? FT=True, AB=True

spe

## Step 9: Additional Test Cases (Optional)

Run the same comparison on other Fuster validation DOIs.

In [13]:
# Additional DOIs from fuster_test_extraction_evaluation.ipynb
ADDITIONAL_DOIS = [
    ("10.5061/dryad.2n5h6", "10.1093/jhered/esw073"),  # black-legged tick  
    ("10.5061/dryad.4767v", "10.1002/ece3.1476"),      # Ericaceae
    ("10.5061/dryad.4k275", "10.1186/s40462-016-0079-4"),  # caribou
    ("10.5061/dryad.67t23", "10.1098/rspb.2014.0649"),   # tree swallow
]

print("Additional test DOIs available:")
for dataset_doi, article_doi in ADDITIONAL_DOIS:
    pdf_path = Path("data/pdfs/fuster") / f"{article_doi.replace('/', '_')}.pdf"
    exists = pdf_path.exists()
    size = f"{pdf_path.stat().st_size/1024:.0f}KB" if exists else "N/A"
    print(f"  {dataset_doi} -> {article_doi} (exists={exists}, {size})")

# TODO: Implement batch evaluation if initial test shows promising results

Additional test DOIs available:
  10.5061/dryad.2n5h6 -> 10.1093/jhered/esw073 (exists=True, 8124KB)
  10.5061/dryad.4767v -> 10.1002/ece3.1476 (exists=True, 6300KB)
  10.5061/dryad.4k275 -> 10.1186/s40462-016-0079-4 (exists=True, 2372KB)
  10.5061/dryad.67t23 -> 10.1098/rspb.2014.0649 (exists=True, 670KB)


## Conclusions

### Results Summary

| Metric | Full-text | Abstract-only | Delta |
|--------|-----------|---------------|-------|
| Micro F1 | TBD | TBD | TBD |
| Token Cost | TBD | TBD | TBD |

### Key Findings

1. **TBD** - Does full-text improve extraction?
2. **TBD** - Which fields benefit most?
3. **TBD** - Is cost increase justified?

### Recommendations

TBD after running notebook.